# ADMET Modelling for Drug Risk Assessment
### Predicting Absorption, Distribution, Metabolism, Excretion & Toxicity to Replace Animal Experiments

**Author:** Himanshu Goel | [himanshugoel.github.io](https://himanshugoel.github.io)

---

## What is ADMET and why does it matter?

ADMET describes the fate and safety of a drug candidate in the body:

```
ABSORPTION    How much reaches systemic circulation?
DISTRIBUTION  Where does it go in the body?
METABOLISM    How is it broken down?
EXCRETION     How is it eliminated?
TOXICITY      What harm can it cause?
```

**90% of drug failures occur due to poor ADMET properties.** Computational ADMET models predict
these properties from structure alone, replacing expensive and time-consuming animal experiments.

### Animal experiments replaced by ADMET models

| Animal test | ADMET model replacement | Regulatory acceptance |
|-------------|------------------------|-----------------------|
| Oral bioavailability (rat) | LogS, Papp, P-gp model | FDA, EMA |
| Protein binding (in vivo) | fup prediction (LogP/MW) | EPA HTTK |
| Metabolic stability (hepatocyte) | CLint QSAR | OECD GD 69 |
| hERG / QT in vivo | hERG QSAR + CiPA | ICH E14/S7B 2022 |
| Ames mutagenicity | QSAR RF/GNN (ICH M7) | ICH M7(R2) |
| Acute oral toxicity (LD50) | QSAR regression | OECD TG 471 NAM |
| Skin sensitisation (LLNA) | In silico DA | OECD TG 497 |
| DILI (28-day rat) | Liver organoid + QSAR | FDA Modernization Act 2.0 |
| BBB penetration (in vivo) | CNS MPO / pKa-LogP | CNS drug design |
| Volume of distribution (rat) | QSAR Vd | PK guidance |

## Tutorial structure

| Section | ADMET endpoint | ML method |
|---------|---------------|-----------|
| 1 | Setup and dataset | RDKit descriptors |
| 2 | Descriptor engineering | 200+ physicochemical features |
| 3 | Absorption: Caco-2, Papp, LogS | Random Forest / XGBoost |
| 4 | Distribution: fup, BBB, Vd | Gradient Boosting |
| 5 | Metabolism: CLint, CYP inhibition | Multi-output models |
| 6 | Excretion: renal / hepatic clearance | Ridge regression |
| 7 | Toxicity: hERG, Ames, DILI, LD50 | Ensemble + GNN template |
| 8 | Multi-endpoint ADMET model | Stacked ensemble |
| 9 | Applicability domain and UQ | Tanimoto AD, conformal pred. |
| 10 | Drug risk assessment pipeline | IVIVE + 3Rs report |

---
## Section 1 -- Setup, Imports & Molecular Dataset

We build a curated ADMET dataset of 50 reference drugs with known experimental values,
drawn from literature and public databases (ChEMBL, DrugBank, FDA labels).

In [ ]:
# !pip install rdkit scikit-learn xgboost lightgbm shap matplotlib seaborn pandas

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import json, re
from collections import Counter

from rdkit import Chem, DataStructs
from rdkit.Chem import (
    Descriptors, rdMolDescriptors, AllChem, QED,
    MolStandardize, FilterCatalog
)
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_predict
from sklearn.metrics import (roc_auc_score, matthews_corrcoef,
                              mean_squared_error, r2_score, mean_absolute_error)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False

np.random.seed(42)
print('All imports OK')
print(f'XGBoost available: {XGB_AVAILABLE}')

In [ ]:
# ── Reference ADMET dataset (50 well-characterised drugs) ──────────────────
# Values from: ChEMBL, DrugBank, FDA labels, published ADMET benchmarks
# Endpoints: LogS (solubility), Papp (Caco-2), fup, CLint, LogD, hERG, Ames

ADMET_DATA = [
    # (name, SMILES, LogS, Papp_Caco2, fup, CLint_mL_min_mg, logD74, BBB, hERG_active, Ames_pos, LD50_mgkg, DILI)
    # LogS: log10(mol/L) | Papp: x10^-6 cm/s | fup: fraction | CLint: mL/min/mg protein
    # BBB: 1=penetrant | hERG: 1=active (IC50<10uM) | Ames: 1=mutagenic | DILI: 1=concern
    ('Aspirin',           'CC(=O)Oc1ccccc1C(=O)O',                    -1.8,  3.5,  0.01, 0.4,  0.3,  0, 0, 0, 200,  0),
    ('Ibuprofen',         'CC(C)Cc1ccc(cc1)C(C)C(=O)O',               -3.5, 15.2,  0.01, 1.2,  3.5,  0, 0, 0, 636,  0),
    ('Naproxen',          'CC(C(=O)O)c1ccc2cccc(OC)c2c1',             -3.9,  8.1,  0.01, 0.3,  2.8,  0, 0, 0, 534,  0),
    ('Diclofenac',        'O=C(O)Cc1ccccc1Nc1c(Cl)cccc1Cl',           -4.2, 11.0,  0.01, 8.5,  3.1,  0, 1, 0, 150,  1),
    ('Acetaminophen',     'CC(=O)Nc1ccc(O)cc1',                       -1.0, 12.6,  0.80, 4.2,  0.5,  0, 0, 0, 338,  1),
    ('Caffeine',          'Cn1cnc2c1c(=O)n(C)c(=O)n2C',               -1.2, 32.1,  0.64, 2.1, -0.1,  1, 0, 0, 367,  0),
    ('Metformin',         'CN(C)C(=N)NC(=N)N',                         0.5,  0.3,  1.00, 0.1, -3.1,  0, 0, 0,1450,  0),
    ('Atorvastatin',      'CC(C)c1c(C(=O)Nc2ccccc2F)c(-c2ccccc2)n(CC[C@@H](O)C[C@@H](O)CC(=O)O)c1-c1ccc(F)cc1', -3.8,  4.2,  0.02, 12.0, 4.5,  0, 1, 0,  34,  0),
    ('Rosuvastatin',      'CC(C)c1nc(N(C)S(=O)(=O)C)nc(-c2ccc(F)cc2)c1/C=C/[C@@H](O)C[C@@H](O)CC(=O)O', -3.1,  0.9,  0.12,  1.8, 0.3,  0, 0, 0,  50,  0),
    ('Omeprazole',        'COc1ccc2[nH]c(/C(=N\\OS(=O)(=O)c3ccccc3)/CC)nc2c1OC', -3.5,  5.6,  0.05, 22.0, 2.4,  1, 0, 0, 720,  0),
    ('Metoprolol',        'CCCOC(=O)c1ccc(OCC(O)CNC(C)C)cc1',         -2.1, 18.9,  0.88, 15.2, 1.6,  1, 0, 0, 750,  0),
    ('Amlodipine',        'CCOC(=O)c1c(COCCN)nc(C)c(C(=O)OCC)c1-c1ccccc1Cl', -3.8,  8.3,  0.03, 5.8,  2.5,  1, 1, 0,  37,  0),
    ('Ciprofloxacin',     'O=C(O)c1cn2c(=O)c(CN3CCNCC3)cc2c2cc(F)ccc12', -3.5,  3.1,  0.60, 2.2, -1.1,  1, 0, 0,2500,  0),
    ('Amoxicillin',       'CC1(C)SC2C(NC1=O)C(=O)N2Cc1ccc(O)cc1',     -2.2,  0.5,  0.82, 1.0, -2.3,  0, 0, 0,2000,  0),
    ('Doxycycline',       'OC1=C(O)C(=O)[C@]2(O)C(=C1)C[C@H](O)[C@]1(O)C(=O)c3c(O)cccc3[C@@]12O', -1.8,  6.5,  0.69, 1.5, -0.2,  0, 0, 0, 900,  0),
    ('Fluoxetine',        'CNCCC(Oc1ccc(cc1)C(F)(F)F)c1ccccc1',       -4.5, 25.8,  0.06, 8.3,  4.0,  1, 1, 0, 248,  0),
    ('Sertraline',        'CNC1CCC(c2ccc(Cl)c(Cl)c2)c2ccccc21',       -5.0, 15.1,  0.02, 22.5, 4.8,  1, 1, 0, 290,  0),
    ('Escitalopram',      'CCOC(=O)c1ccc2cc(CN(C)C)ccc2n1',           -3.5, 14.2,  0.04, 11.8, 3.2,  1, 1, 0, 100,  0),
    ('Alprazolam',        'Cc1nnc2n1-c1ccc(Cl)cc1C(=NCC(=O)c1ccccc1)C=2', -3.8,  9.2,  0.18, 10.5, 2.6,  1, 1, 0, 331,  0),
    ('Lorazepam',         'OC1N=C(c2ccccc2Cl)c2cc(Cl)ccc2NC1=O',      -3.5,  6.8,  0.11, 1.2,  2.4,  1, 1, 0,  90,  0),
    ('Warfarin',          'CC(=O)CC(c1ccccc1)c1c(O)c2ccccc2oc1=O',    -3.2,  3.9,  0.01, 0.4,  2.5,  0, 1, 0, 374,  1),
    ('Simvastatin',       'CCC(C)(C)C(=O)O[C@@H]1C[C@@H](C)C=C2C=C[C@H](O)[C@@H](CC[C@@H](OC(=O)C)C(C)(C)CC)C21', -5.5,  8.5,  0.04, 18.5, 4.5,  0, 0, 0, 940,  1),
    ('Clopidogrel',       'COCC(=O)N1CCCC12c1ccccc1SC2Cc1cccnc12',    -3.8, 15.2,  0.02, 25.0, 3.8,  1, 1, 0, 124,  0),
    ('Tamoxifen',         'CC/C(=C(\\c1ccccc1)/c1ccc(OCCN(C)C)cc1)c1ccccc1', -5.8,  9.8,  0.01, 42.0, 6.5,  1, 1, 0, 338,  1),
    ('Rifampicin',        'COC1C2OC3(C)C=CC1CC2OC(=O)/C(=C/NC4=Nc5c(O)c(O)c(C)cc5C=C4)C', -3.2,  2.8,  0.15, 5.5,  1.8,  1, 0, 0, 885,  1),
    ('Isoniazid',         'NNC(=O)c1ccncc1',                           0.2,  1.5,  0.88, 3.5, -1.2,  0, 0, 0, 940,  1),
    ('Metronidazole',     'CC1=NC=C(CCO)N1C(=O)c1ccc([N+](=O)[O-])cc1', -0.8,  8.5,  0.80, 4.5, -0.2,  0, 0, 1, 600,  0),
    ('Nitrofurantoin',    'O=C1NC(=O)C(=C1/N=N/c1ccc(o1)[N+](=O)[O-])C1=O', -2.8,  2.1,  0.65, 6.2, -0.5,  0, 0, 1, 400,  0),
    ('Valproic acid',     'CCCC(CCC)C(=O)O',                          -2.2, 14.8,  0.10, 0.8,  2.0,  0, 0, 0, 670,  0),
    ('Carbamazepine',     'NC(=O)N1c2ccccc2C=Cc2ccccc21',             -3.2,  7.5,  0.24, 3.5,  2.2,  1, 0, 0, 572,  0),
    ('Phenytoin',         'O=C1NC(=O)C(c2ccccc2)(c2ccccc2)N1',        -4.5,  5.8,  0.08, 1.2,  2.8,  1, 1, 0, 100,  0),
    ('Atorvastatin',      'CC(C)c1c(C(=O)Nc2ccccc2F)c(-c2ccccc2)n(CCC(O)CC(O)CC(=O)O)c1-c1ccc(F)cc1', -3.8,  4.2,  0.02, 12.0, 4.5,  0, 1, 0,  34,  0),
    ('Sildenafil',        'CCCC1=NN(C)C(=O)c2cc(cnc21)c1ccc(cc1)S(=O)(=O)N1CCN(C)CC1', -3.8,  9.5,  0.04, 18.5, 2.2,  1, 1, 0, 200,  0),
    ('Celecoxib',         'Cc1ccc(-c2cc(C(F)(F)F)nn2-c2ccc(S(N)(=O)=O)cc2)cc1', -5.5,  6.2,  0.03, 8.8,  3.5,  0, 0, 0, 360,  0),
    ('Meloxicam',         'Cc1cnc(NC(=O)c2cccc(S(=O)(=O)Nc3ncc(C)s3)c2O)s1', -5.2,  3.8,  0.01, 0.4,  1.8,  0, 1, 0, 470,  0),
    ('Tramadol',          'OC12CC3CC(CC(C3)C1)CC2C(c1ccccc1)N(C)C',   -2.5, 21.5,  0.81, 12.5, 1.5,  1, 1, 0, 228,  0),
    ('Amitriptyline',     'CN(C)CCC=C1c2ccccc2CCc2ccccc21',           -4.5, 22.8,  0.05, 18.5, 4.2,  1, 1, 0, 111,  0),
    ('Haloperidol',       'OC1(CCc2ccc(Cl)cc2)CCN(CCCC(=O)c2ccc(F)cc2)CC1', -4.8,  9.8,  0.07, 12.2, 3.5,  1, 1, 0, 165,  0),
    ('Chlorpromazine',    'CN(C)CCCN1c2ccccc2Sc2ccc(Cl)cc21',         -5.2, 18.5,  0.05, 22.5, 4.8,  1, 1, 0,  84,  0),
    ('Captopril',         'CC(CS)C(=O)N1CCCC1C(=O)O',                 -1.2,  0.8,  0.90, 3.5, -0.8,  0, 0, 0,6000,  0),
    ('Lisinopril',        'NCCCC(N)C(=O)N1CCCC1C(=O)O',               -1.5,  0.2,  0.70, 1.5, -3.5,  0, 0, 0,5000,  0),
    ('Furosemide',        'NS(=O)(=O)c1cc(C(=O)O)c(NCc2ccco2)cc1Cl',  -3.8,  1.2,  0.02, 0.5, -0.5,  0, 0, 0, 860,  0),
    ('Hydrochlorothiazide','NS(=O)(=O)c1cc2c(cc1Cl)NCNS2(=O)=O',      -3.2,  0.5,  0.59, 0.8, -1.2,  0, 0, 0,  10,  0),
    ('Dexamethasone',     'C[C@@H]1C[C@H]2[C@@H](CCC3=CC(=O)C=C[C@@]23C)[C@@]1(O)C(=O)CO', -2.8,  5.8,  0.23, 22.5, 1.8,  1, 0, 0, 210,  0),
    ('Prednisolone',      'OCC(=O)[C@@H]1CC[C@H]2[C@@H]3CCC4=CC(=O)C=C[C@]4(C)[C@@H]3[C@@H](O)C[C@@]12C', -2.4,  4.5,  0.24, 15.2, 1.5,  1, 0, 0, 700,  0),
    ('Bisphenol A',       'CC(C)(c1ccc(O)cc1)c1ccc(O)cc1',            -3.5, 18.5,  0.14, 8.5,  3.8,  0, 0, 0, 3250, 1),
    ('Paclitaxel',        'CC(=O)O[C@@H]1[C@H](OC(=O)c2ccccc2)[C@]2(O)C[C@H]3OC[C@@]3(OC(C)=O)[C@@H](O)[C@@H]2[C@H]1c1ccccc1', -5.5,  0.5,  0.12, 8.0,  3.2,  0, 0, 0,  23,  1),
    ('Doxorubicin',       'COc1cccc2C(=O)c3c(O)c4c(c(O)c3C(=O)c12)C[C@](O)(C(=O)CO)C[C@@H]4N', -1.8,  1.2,  0.25, 15.5, -0.5,  0, 0, 0,  21,  1),
    ('Methotrexate',      'CN(Cc1cnc2nc(N)nc(N)c2n1)c1ccc(C(=O)N[C@@H](CCC(=O)O)C(=O)O)cc1', -3.2,  0.5,  0.55, 2.5, -2.5,  0, 0, 0, 180,  1),
    ('Cisplatin',         '[NH3][Pt](Cl)(Cl)[NH3]',                     1.5,  0.5,  0.25, 5.0, -2.5,  0, 0, 0,   6,  1),
]

COLS = ['name','smiles','logS','Papp_Caco2','fup','CLint','logD74','BBB','hERG','Ames','LD50_mgkg','DILI']
df = pd.DataFrame(ADMET_DATA, columns=COLS)

print(f'Dataset: {len(df)} compounds')
print(f'\nEndpoints:')
for col in ['logS','Papp_Caco2','fup','CLint','logD74']:
    print(f'  {col:12s}: min={df[col].min():.2f}  max={df[col].max():.2f}  mean={df[col].mean():.2f}')
for col in ['BBB','hERG','Ames','DILI']:
    print(f'  {col:12s}: positives={df[col].sum()} ({df[col].mean()*100:.0f}%)')
print(f'  LD50     : min={df["LD50_mgkg"].min()}  max={df["LD50_mgkg"].max()}  mean={df["LD50_mgkg"].mean():.0f} mg/kg')

---
## Section 2 -- Molecular Descriptor Engineering

Descriptors encode chemical structure as numerical features. We compute **200+ features**
across four descriptor classes:

| Class | Examples | Count |
|-------|---------|-------|
| Physicochemical | MW, LogP, TPSA, HBD, HBA, Fsp3, QED | 25 |
| Topological | Rings, rotatable bonds, complexity | 15 |
| Morgan fingerprints | ECFP4 (2048-bit) | 2048 |
| MACCS keys | 166 structural keys | 166 |
| RDKit fingerprints | 2048-bit path-based | 2048 |

For most ADMET models, **physicochemical + Morgan fingerprints** give the best balance
of interpretability and predictive power.

In [ ]:
# ── Standardise all SMILES first ────────────────────────────────────────────
def standardise(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    mol = rdMolStandardize.LargestFragmentChooser().choose(mol)
    mol = rdMolStandardize.Uncharger().uncharge(mol)
    return mol

df['mol'] = df['smiles'].apply(standardise)
valid = df['mol'].notna()
print(f'Valid molecules: {valid.sum()}/{len(df)}')

# ── Physicochemical descriptor calculator ─────────────────────────────────
def physchem_descriptors(mol):
    if mol is None: return [None]*26
    try:
        mw    = Descriptors.MolWt(mol)
        logp  = Descriptors.MolLogP(mol)
        tpsa  = Descriptors.TPSA(mol)
        hbd   = rdMolDescriptors.CalcNumHBD(mol)
        hba   = rdMolDescriptors.CalcNumHBA(mol)
        rb    = rdMolDescriptors.CalcNumRotatableBonds(mol)
        rings = rdMolDescriptors.CalcNumRings(mol)
        aro_r = rdMolDescriptors.CalcNumAromaticRings(mol)
        ali_r = rdMolDescriptors.CalcNumAliphaticRings(mol)
        fsp3  = rdMolDescriptors.CalcFractionCSP3(mol)
        hetero= rdMolDescriptors.CalcNumHeteroatoms(mol)
        stereo= rdMolDescriptors.CalcNumStereocenters(mol)
        heavy = mol.GetNumHeavyAtoms()
        mwlog = np.log10(max(1,mw))
        qed_v = QED.qed(mol)
        chiral= rdMolDescriptors.CalcNumAtomStereoCenters(mol)
        n_N   = sum(1 for a in mol.GetAtoms() if a.GetAtomicNum()==7)
        n_O   = sum(1 for a in mol.GetAtoms() if a.GetAtomicNum()==8)
        n_S   = sum(1 for a in mol.GetAtoms() if a.GetAtomicNum()==16)
        n_hal = sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() in (9,17,35,53))
        n_basicN = sum(1 for a in mol.GetAtoms()
                       if a.GetAtomicNum()==7 and a.GetTotalNumHs()>0)
        ro5_v = sum([mw>500, logp>5, hbd>5, hba>10])
        mw2   = mw*mw
        logp2 = logp*logp
        tpsa_logp = tpsa * logp
        psa_mw = tpsa / max(1, mw) * 100
        return [mw, logp, tpsa, hbd, hba, rb, rings, aro_r, ali_r, fsp3,
                hetero, stereo, heavy, mwlog, qed_v, chiral, n_N, n_O, n_S,
                n_hal, n_basicN, ro5_v, mw2, logp2, tpsa_logp, psa_mw]
    except: return [None]*26

PHYSCHEM_COLS = [
    'MW','LogP','TPSA','HBD','HBA','RotBonds','Rings','ArRings','AliRings',
    'Fsp3','Heteroatoms','Stereocenters','HeavyAtoms','LogMW','QED','Chiral',
    'nN','nO','nS','nHal','BasicN','Ro5_violations','MW2','LogP2','TPSA_LogP','PSA_MW'
]

physchem_vals = [physchem_descriptors(mol) for mol in df['mol']]
df_phys = pd.DataFrame(physchem_vals, columns=PHYSCHEM_COLS)
print(f'Physicochemical descriptors: {df_phys.shape[1]}')
print(df_phys.head(3).to_string())

In [ ]:
# ── Fingerprint descriptors ─────────────────────────────────────────────────
def morgan_fp(mol, radius=2, n_bits=2048):
    if mol is None: return np.zeros(n_bits)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    return np.array(fp, dtype=np.uint8)

def maccs_fp(mol):
    if mol is None: return np.zeros(167)
    from rdkit.Chem import MACCSkeys
    fp = MACCSkeys.GenMACCSKeys(mol)
    return np.array(fp, dtype=np.uint8)

def rdkit_fp(mol, n_bits=2048):
    if mol is None: return np.zeros(n_bits)
    from rdkit.Chem import RDKFingerprint
    fp = RDKFingerprint(mol, fpSize=n_bits)
    return np.array(fp, dtype=np.uint8)

# Compute all fingerprints
morgan_fps = np.array([morgan_fp(mol) for mol in df['mol']])  # [50, 2048]
maccs_fps  = np.array([maccs_fp(mol)  for mol in df['mol']])  # [50, 167]

print(f'Morgan ECFP4:   {morgan_fps.shape}')
print(f'MACCS keys:     {maccs_fps.shape}')
print(f'Mean bit density (Morgan): {morgan_fps.mean():.4f}')
print(f'Mean bit density (MACCS):  {maccs_fps.mean():.4f}')

# ── Build feature matrices for different model types ────────────────────────
X_phys   = df_phys.fillna(0).values.astype(np.float32)
X_morgan = morgan_fps.astype(np.float32)
X_maccs  = maccs_fps.astype(np.float32)
X_hybrid = np.hstack([X_phys, X_maccs])  # hybrid: physchem + MACCS (fast models)
X_full   = np.hstack([X_phys, X_morgan]) # full: physchem + Morgan (best models)

print(f'\nFeature matrix shapes:')
print(f'  Physicochemical only: {X_phys.shape}')
print(f'  Morgan ECFP4 only:    {X_morgan.shape}')
print(f'  Hybrid (phys+MACCS):  {X_hybrid.shape}')
print(f'  Full (phys+Morgan):   {X_full.shape}')

In [ ]:
# ── 2.1 Feature correlation and importance analysis ─────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Physicochemical descriptor distributions for drugs
ax = axes[0]
key_props = ['MW','LogP','TPSA','HBD','HBA','QED']
vals = [df_phys[p].dropna() for p in key_props]
ax.boxplot(vals, labels=key_props, patch_artist=True,
           boxprops=dict(facecolor='#1565C0', alpha=0.6),
           medianprops=dict(color='white', lw=2))
ax.set_ylabel('Value'); ax.set_title('Physicochemical Distribution\n(50 reference drugs)', fontweight='bold')
ax.grid(True, alpha=0.3)

# Plot 2: ADMET endpoint distributions
ax = axes[1]
endpoints = ['logS','Papp_Caco2','fup','CLint','logD74']
norm_vals = [(df[ep] - df[ep].mean()) / df[ep].std() for ep in endpoints]
ax.boxplot(norm_vals, labels=endpoints, patch_artist=True,
           boxprops=dict(facecolor='#27AE60', alpha=0.6),
           medianprops=dict(color='white', lw=2))
ax.set_ylabel('Z-score'); ax.set_title('ADMET Endpoint Distributions\n(z-scored)', fontweight='bold')
ax.grid(True, alpha=0.3)

# Plot 3: Chemical space PCA
from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=42)
emb = pca.fit_transform(X_full)
sc  = axes[2].scatter(emb[:,0], emb[:,1],
                       c=df['logD74'], cmap='RdBu_r', s=80, alpha=0.85, edgecolors='k', lw=0.5)
plt.colorbar(sc, ax=axes[2], label='logD7.4')
# Label outliers
for i, name in enumerate(df['name']):
    if abs(emb[i,0]) > 3 or abs(emb[i,1]) > 3:
        axes[2].annotate(name[:8], (emb[i,0], emb[i,1]), fontsize=7)
axes[2].set_xlabel('PC1'); axes[2].set_ylabel('PC2')
axes[2].set_title(f'Chemical Space (PCA)\n({pca.explained_variance_ratio_[:2].sum()*100:.0f}% variance)', fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.suptitle('ADMET Dataset Overview', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Section 3 -- Absorption Models: LogS, Papp (Caco-2), P-glycoprotein

Absorption models predict how much drug reaches systemic circulation after oral dosing.
These replace expensive in vitro cell-based assays and in vivo animal studies.

| Model | Endpoint | Replaces | Threshold |
|-------|---------|----------|----------|
| LogS (ESOL) | Aqueous solubility | Shake-flask, nephelometry | LogS > -4 = acceptable |
| Papp Caco-2 | Intestinal permeability | Caco-2 cell monolayer assay | Papp > 10 x10^-6 = high |
| P-gp substrate | Efflux transporter | MDR1-MDCKII assay | Binary classification |
| Bioavailability | %F oral | Rat PK study | %F > 20% = acceptable |

**Lipinski's Ro5** is the simplest absorption filter: MW<=500, LogP<=5, HBD<=5, HBA<=10.

In [ ]:
# ── 3.1 LogS (aqueous solubility) model ────────────────────────────────────
# LogS is critical for formulation and bioavailability
# ESOL model (Delaney 2004) as baseline; RF for improved accuracy

from sklearn.model_selection import cross_val_predict, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

y_logS = df['logS'].values
cv_reg = KFold(n_splits=5, shuffle=True, random_state=42)

# ── ESOL baseline (linear model on 4 descriptors) ─────────────────────────
# LogS = 0.16 - 0.63*cLogP - 0.0062*MW + 0.066*RB - 0.74*AP
# AP = aromatic proportion = ArC / HeavyAtoms
esol_features = np.column_stack([
    df_phys['LogP'].fillna(0),
    df_phys['MW'].fillna(0),
    df_phys['RotBonds'].fillna(0),
    (df_phys['ArRings'].fillna(0) * 6) / df_phys['HeavyAtoms'].replace(0,1),
])
from sklearn.linear_model import LinearRegression
esol_pred = cross_val_predict(LinearRegression(), esol_features, y_logS, cv=cv_reg)
esol_r2   = r2_score(y_logS, esol_pred)
esol_rmse = np.sqrt(mean_squared_error(y_logS, esol_pred))

# ── Random Forest model ─────────────────────────────────────────────────────
rf_logs = RandomForestRegressor(n_estimators=200, min_samples_leaf=2, random_state=42)
rf_pred = cross_val_predict(rf_logs, X_full, y_logS, cv=cv_reg)
rf_r2   = r2_score(y_logS, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_logS, rf_pred))

# ── Gradient Boosting model ──────────────────────────────────────────────────
gb_logs = GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42)
gb_pred = cross_val_predict(gb_logs, X_full, y_logS, cv=cv_reg)
gb_r2   = r2_score(y_logS, gb_pred)
gb_rmse = np.sqrt(mean_squared_error(y_logS, gb_pred))

print('LogS (Aqueous Solubility) Models:')
print(f'  ESOL baseline:       R2={esol_r2:.3f}  RMSE={esol_rmse:.3f} log units')
print(f'  Random Forest:       R2={rf_r2:.3f}  RMSE={rf_rmse:.3f} log units')
print(f'  Gradient Boosting:   R2={gb_r2:.3f}  RMSE={gb_rmse:.3f} log units')
print(f'\nBenchmark: Published ESOL R2 ~0.74 (Delaney 2004), RF typically 0.80-0.88')

# Solubility classes
def solubility_class(logS):
    if logS > -2:   return 'High'
    elif logS > -4: return 'Moderate'
    elif logS > -6: return 'Low'
    else:           return 'Very Low'

df['solubility_class'] = df['logS'].apply(solubility_class)
print('\nSolubility distribution:')
print(df['solubility_class'].value_counts())

In [ ]:
# ── 3.2 Papp Caco-2 permeability model ─────────────────────────────────────
# Papp > 10 x10^-6 cm/s = high permeability (BCS Class I/II)
# Key descriptors: MW, TPSA, LogP, HBD (Veber rules: TPSA<=140, RotBonds<=10)

y_papp = df['Papp_Caco2'].values

# Classification: high vs low permeability
y_papp_class = (y_papp >= 10).astype(int)
print(f'High Papp (>=10): {y_papp_class.sum()}/{len(y_papp_class)} ({y_papp_class.mean()*100:.0f}%)')

cv_cls = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# RF classifier
rf_papp = RandomForestClassifier(n_estimators=200, min_samples_leaf=1, random_state=42)
papp_pred_proba = cross_val_predict(rf_papp, X_full, y_papp_class, cv=cv_cls, method='predict_proba')[:,1]
papp_pred       = (papp_pred_proba >= 0.5).astype(int)
papp_auc        = roc_auc_score(y_papp_class, papp_pred_proba)
papp_mcc        = matthews_corrcoef(y_papp_class, papp_pred)

print(f'Caco-2 Permeability Classifier (RF):')
print(f'  AUC: {papp_auc:.3f}  MCC: {papp_mcc:.3f}')

# Also regression model for exact Papp
rf_papp_reg = RandomForestRegressor(n_estimators=200, min_samples_leaf=2, random_state=42)
papp_reg_pred = cross_val_predict(rf_papp_reg, X_full, y_papp, cv=cv_reg)
papp_r2   = r2_score(y_papp, papp_reg_pred)
papp_rmse = np.sqrt(mean_squared_error(y_papp, papp_reg_pred))
print(f'  Regression R2: {papp_r2:.3f}  RMSE: {papp_rmse:.2f} x10^-6 cm/s')

# Veber rules check
df['veber_pass'] = ((df_phys['TPSA'] <= 140) & (df_phys['RotBonds'] <= 10)).values
print(f'\nVeber rules pass: {df["veber_pass"].sum()}/{len(df)}')
print(f'Concordance Veber vs Papp: {((df["veber_pass"]) == (y_papp_class==1)).mean()*100:.0f}%')

In [ ]:
# ── 3.3 Absorption visualisation ─────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Observed vs Predicted LogS
ax = axes[0]
ax.scatter(y_logS, rf_pred, c='#1565C0', s=70, alpha=0.8, edgecolors='k', lw=0.5)
lims = [min(y_logS.min(), rf_pred.min())-0.3, max(y_logS.max(), rf_pred.max())+0.3]
ax.plot(lims, lims, 'k--', lw=1.5, alpha=0.6, label='Perfect')
ax.set_xlabel('Observed logS (log mol/L)'); ax.set_ylabel('Predicted logS')
ax.set_title(f'LogS Prediction (RF)\nR2={rf_r2:.3f}  RMSE={rf_rmse:.3f}', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

# Plot 2: Caco-2 predicted probability vs measured
ax = axes[1]
colours = ['#E74C3C' if c==1 else '#1565C0' for c in y_papp_class]
ax.scatter(y_papp, papp_pred_proba, c=colours, s=70, alpha=0.8, edgecolors='k', lw=0.5)
ax.axhline(0.5, color='k', linestyle='--', lw=1.5, alpha=0.6, label='Decision boundary')
ax.axvline(10,  color='green', linestyle='--', lw=1.5, alpha=0.6, label='Papp=10 threshold')
ax.set_xlabel('Measured Papp (x10^-6 cm/s)'); ax.set_ylabel('P(High Papp)')
ax.set_title(f'Caco-2 Classifier\nAUC={papp_auc:.3f}  MCC={papp_mcc:.3f}', fontweight='bold')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Plot 3: Absorption space (Biopharmaceutics)
ax = axes[2]
sc = ax.scatter(df['logD74'], df['logS'],
                c=y_papp_class, cmap='RdYlGn', s=80, alpha=0.85, vmin=0, vmax=1)
ax.axvline(3, color='k', lw=1, linestyle='--', alpha=0.5)
ax.axhline(-4, color='k', lw=1, linestyle='--', alpha=0.5)
ax.set_xlabel('logD7.4 (lipophilicity)'); ax.set_ylabel('logS (solubility)')
ax.set_title('BCS Chemical Space\n(green=high Papp, red=low)', fontweight='bold')
plt.colorbar(sc, ax=ax, label='High Papp')
for i, row in df.iterrows():
    if df_phys.loc[i,'QED'] > 0.8:
        ax.annotate(df.loc[i,'name'][:6], (df.loc[i,'logD74'], df.loc[i,'logS']), fontsize=7)
ax.grid(True, alpha=0.3)

plt.suptitle('Absorption Models: LogS and Caco-2 Permeability', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Section 4 -- Distribution Models: fup, BBB, Volume of Distribution

Distribution models predict where the drug goes after absorption.

| Model | Endpoint | Animal test replaced | Key descriptors |
|-------|---------|---------------------|----------------|
| fup | Plasma protein binding | Ultracentrifugation (rabbit) | LogP, MW, pKa proxy |
| BBB | Blood-brain barrier penetration | Rat brain Kp study | CNS MPO score |
| Vd | Volume of distribution | Rat IV PK study | LogP, fup, pKa |

In [ ]:
# ── 4.1 Fraction unbound in plasma (fup) ────────────────────────────────────
# fup determines the free (pharmacologically active) drug concentration
# Critical for IVIVE: Css_free = Css_total x fup

y_fup = df['fup'].values

# Log-transform fup for better model performance (right-skewed distribution)
y_fup_log = np.log10(y_fup.clip(0.001, 1.0))

# Model: fup is primarily driven by lipophilicity and molecular weight
# Lobell (2003): log(fup) = -0.028*LogP - 0.0038*MW + 1.2
logp_vals = df_phys['LogP'].fillna(0).values
mw_vals   = df_phys['MW'].fillna(0).values

# Lobell baseline
lobell_pred_log = -0.028*logp_vals - 0.0038*mw_vals + 1.2
lobell_pred     = np.clip(10**lobell_pred_log, 0.001, 1.0)
lobell_rmse     = np.sqrt(mean_squared_error(y_fup_log, lobell_pred_log))
lobell_r2       = r2_score(y_fup_log, lobell_pred_log)

# Random Forest
rf_fup   = RandomForestRegressor(n_estimators=200, min_samples_leaf=2, random_state=42)
fup_pred_log = cross_val_predict(rf_fup, X_full, y_fup_log, cv=KFold(5, shuffle=True, random_state=42))
fup_r2   = r2_score(y_fup_log, fup_pred_log)
fup_rmse = np.sqrt(mean_squared_error(y_fup_log, fup_pred_log))

print('fup (Plasma Protein Binding) Models:')
print(f'  Lobell baseline:  R2={lobell_r2:.3f}  RMSE={lobell_rmse:.3f} log units')
print(f'  Random Forest:    R2={fup_r2:.3f}  RMSE={fup_rmse:.3f} log units')

# ── 4.2 BBB penetration classifier ─────────────────────────────────────────
# CNS MPO (Pfizer, Wager 2010): 6 properties scored 0-1
# MW<=360, cLogP 1-3, TPSA<=90, HBD<=1, pKa proxy, arRings<=2

y_bbb = df['BBB'].values
print(f'\nBBB penetrant: {y_bbb.sum()}/{len(y_bbb)} ({y_bbb.mean()*100:.0f}%)')

def cns_mpo_score(mol):
    if mol is None: return 0
    mw   = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    tpsa = Descriptors.TPSA(mol)
    hbd  = rdMolDescriptors.CalcNumHBD(mol)
    naro = rdMolDescriptors.CalcNumAromaticRings(mol)
    score = sum([
        1 if mw <= 360 else (0 if mw >= 500 else (500-mw)/140),
        1 if 1 <= logp <= 3 else (0 if logp < 0 or logp > 5 else 0.5),
        1 if tpsa <= 90 else (0 if tpsa >= 120 else (120-tpsa)/30),
        1 if hbd == 0 else (0 if hbd >= 2 else 0.5),
        1 if logp < 5 else 0,
        1 if naro <= 2 else (0 if naro >= 4 else 0.5),
    ])
    return score / 6  # normalise to 0-1

df['CNS_MPO'] = [cns_mpo_score(mol) for mol in df['mol']]

# RF BBB classifier
rf_bbb = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
bbb_proba = cross_val_predict(rf_bbb, X_full, y_bbb, cv=StratifiedKFold(5, shuffle=True, random_state=42), method='predict_proba')[:,1]
bbb_auc = roc_auc_score(y_bbb, bbb_proba)
bbb_mcc = matthews_corrcoef(y_bbb, (bbb_proba>=0.5).astype(int))

# CNS MPO baseline
mpo_auc = roc_auc_score(y_bbb, df['CNS_MPO'])

print(f'BBB Classification:')
print(f'  CNS MPO baseline: AUC={mpo_auc:.3f}')
print(f'  Random Forest:    AUC={bbb_auc:.3f}  MCC={bbb_mcc:.3f}')

In [ ]:
# ── 4.3 Distribution visualisation ─────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Observed vs Predicted fup (log scale)
ax = axes[0]
ax.scatter(y_fup_log, fup_pred_log, c='#1565C0', s=70, alpha=0.8, edgecolors='k', lw=0.5)
lims = [y_fup_log.min()-0.2, y_fup_log.max()+0.2]
ax.plot(lims, lims, 'k--', lw=1.5, alpha=0.6)
ax.set_xlabel('Observed log10(fup)'); ax.set_ylabel('Predicted log10(fup)')
ax.set_title(f'fup Prediction (RF)\nR2={fup_r2:.3f}', fontweight='bold')
ax.grid(True, alpha=0.3)

# Plot 2: BBB probability vs CNS MPO
ax = axes[1]
cols = ['#E74C3C' if b==1 else '#1565C0' for b in y_bbb]
ax.scatter(df['CNS_MPO'], bbb_proba, c=cols, s=80, alpha=0.85)
ax.axhline(0.5, color='k', linestyle='--', lw=1.5, alpha=0.5)
ax.axvline(0.67, color='green', linestyle='--', lw=1.5, alpha=0.5, label='CNS MPO >= 4/6')
ax.set_xlabel('CNS MPO Score (0-1)'); ax.set_ylabel('P(BBB penetrant)')
ax.set_title(f'BBB Classifier\nAUC={bbb_auc:.3f}', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Plot 3: LogP vs fup coloured by BBB
ax = axes[2]
bbb_cols = ['#E74C3C' if b==1 else '#1565C0' for b in y_bbb]
ax.scatter(logp_vals, y_fup, c=bbb_cols, s=80, alpha=0.8)
ax.set_yscale('log'); ax.set_ylabel('fup (log scale)')
ax.set_xlabel('LogP'); ax.set_title('LogP vs fup\n(red=BBB penetrant)', fontweight='bold')
ax.axhline(0.1, color='k', lw=1, linestyle='--', alpha=0.4, label='fup=0.10')
ax.axvline(2, color='green', lw=1, linestyle='--', alpha=0.4)
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('Distribution Models: fup and BBB Penetration', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Section 5 -- Metabolism Models: CLint, CYP Inhibition

Metabolism models predict hepatic clearance and CYP enzyme interactions,
replacing microsomal stability assays and hepatocyte incubations.

| Model | Endpoint | Animal test replaced | Regulatory use |
|-------|---------|---------------------|----------------|
| CLint | Intrinsic hepatic clearance | HLM microsomal stability | EPA HTTK IVIVE |
| CYP3A4 inhibition | Major metabolism enzyme | Midazolam cocktail assay | DDI FDA guidance |
| CYP2D6 inhibition | Secondary CYP | Dextromethorphan assay | ICH M12 |
| Metabolic stability class | High/medium/low | HLM T1/2 | Tier 1 screen |

In [ ]:
# ── 5.1 CLint (intrinsic hepatic clearance) model ──────────────────────────
# CLint (mL/min/mg microsomal protein) predicts hepatic first-pass extraction
# Critical for IVIVE: CLh = Qh * CLint * fup / (Qh + CLint * fup)

y_clint = df['CLint'].values
y_clint_log = np.log10(y_clint.clip(0.01))

cv_reg = KFold(n_splits=5, shuffle=True, random_state=42)

rf_clint  = RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=42)
gb_clint  = GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)

clint_rf_pred = cross_val_predict(rf_clint, X_full, y_clint_log, cv=cv_reg)
clint_gb_pred = cross_val_predict(gb_clint, X_full, y_clint_log, cv=cv_reg)

rf_r2   = r2_score(y_clint_log, clint_rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_clint_log, clint_rf_pred))
gb_r2   = r2_score(y_clint_log, clint_gb_pred)
gb_rmse = np.sqrt(mean_squared_error(y_clint_log, clint_gb_pred))

print('CLint Prediction Models:')
print(f'  Random Forest:      R2={rf_r2:.3f}  RMSE={rf_rmse:.3f} log units')
print(f'  Gradient Boosting:  R2={gb_r2:.3f}  RMSE={gb_rmse:.3f} log units')

# Metabolic stability classification
def clint_class(clint):
    if clint < 3:    return 'Stable'     # t1/2 > 30 min HLM
    elif clint < 20: return 'Moderate'
    else:            return 'Unstable'   # t1/2 < 15 min HLM

df['metab_class'] = df['CLint'].apply(clint_class)
print('\nMetabolic stability:')
print(df['metab_class'].value_counts())

# ── 5.2 CYP inhibition (simulated) ──────────────────────────────────────────
# Known CYP3A4/2D6 inhibitors from DrugBank
np.random.seed(42)
CYP3A4_inhib = np.array([
    0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,1,1,1,0,0,0,1,1,1,0,0,0,0,0,
    0,1,1,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0
])
CYP2D6_inhib = np.array([
    0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,
    0,0,1,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0
])
print(f'\nCYP3A4 inhibitors: {CYP3A4_inhib.sum()}/{len(CYP3A4_inhib)}')
print(f'CYP2D6 inhibitors: {CYP2D6_inhib.sum()}/{len(CYP2D6_inhib)}')

# CYP3A4 classifier
rf_cyp3a4 = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
cyp3a4_prob = cross_val_predict(rf_cyp3a4, X_full, CYP3A4_inhib,
                                 cv=StratifiedKFold(5, shuffle=True, random_state=42),
                                 method='predict_proba')[:,1]
cyp_auc = roc_auc_score(CYP3A4_inhib, cyp3a4_prob)
print(f'CYP3A4 Inhibition Classifier AUC: {cyp_auc:.3f}')

---
## Section 6 -- Toxicity Models: hERG, Ames, DILI, LD50

Toxicity models are the highest-value ADMET predictions for animal experiment reduction.

| Endpoint | Model type | Regulatory framework | Accuracy target |
|----------|-----------|---------------------|----------------|
| hERG IC50 | Regression + classifier | ICH E14/S7B, CiPA | AUC > 0.85 |
| Ames mutagenicity | Classifier | ICH M7(R2) two-method | Sensitivity >= 90% |
| DILI risk | Classifier | FDA Modernization Act 2.0 | AUC > 0.80 |
| LD50 (acute) | Regression | OECD TG 401 NAM | RMSE < 0.5 log units |
| Skin sensitisation | Classifier | OECD TG 497 DA2 | Sensitivity >= 80% |

In [ ]:
# ── 6.1 hERG cardiotoxicity model ────────────────────────────────────────────
# hERG IC50 < 10 uM = active (cardiac safety flag)
# CiPA requires IC50 for 7 channels; hERG is most critical

y_herg = df['hERG'].values
print(f'hERG active: {y_herg.sum()}/{len(y_herg)} ({y_herg.mean()*100:.0f}%)')

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# hERG known structural drivers:
# - High lipophilicity (LogP > 2)
# - Basic nitrogen (pKaB ~ 8-10)
# - Aromatic rings
# - MW 300-600

# Multi-model comparison
models_herg = {
    'Logistic Regression': LogisticRegression(C=0.5, max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=300, min_samples_leaf=1, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, max_depth=4, random_state=42),
    'Extra Trees':         ExtraTreesClassifier(n_estimators=300, min_samples_leaf=1, random_state=42),
}
herg_results = {}
print(f'\n{"Model":22s} {"AUC":>8} {"MCC":>8} {"Sens":>8} {"Spec":>8}')
print('-'*55)
for name, clf in models_herg.items():
    proba = cross_val_predict(clf, X_full, y_herg, cv=cv5, method='predict_proba')[:,1]
    pred  = (proba >= 0.5).astype(int)
    auc   = roc_auc_score(y_herg, proba)
    mcc   = matthews_corrcoef(y_herg, pred)
    tp    = ((pred==1)&(y_herg==1)).sum()
    fn    = ((pred==0)&(y_herg==1)).sum()
    tn    = ((pred==0)&(y_herg==0)).sum()
    fp    = ((pred==1)&(y_herg==0)).sum()
    sens  = tp/(tp+fn) if (tp+fn)>0 else 0
    spec  = tn/(tn+fp) if (tn+fp)>0 else 0
    herg_results[name] = {'auc':auc,'mcc':mcc,'sens':sens,'spec':spec,'proba':proba}
    print(f'{name:22s} {auc:8.3f} {mcc:8.3f} {sens:8.3f} {spec:8.3f}')

best_herg = max(herg_results, key=lambda k: herg_results[k]['auc'])
print(f'\nBest model: {best_herg} (AUC={herg_results[best_herg]["auc"]:.3f})')

In [ ]:
# ── 6.2 Ames mutagenicity (ICH M7) ─────────────────────────────────────────
# ICH M7(R2) requires sensitivity >= 90% for regulatory acceptance
# This is the most safety-critical ADMET model

y_ames = df['Ames'].values
print(f'Ames positive: {y_ames.sum()}/{len(y_ames)} ({y_ames.mean()*100:.0f}%)')

# ICH M7 structural alerts (SMARTS-based) as baseline
ICH_M7_SMARTS = [
    ('Nitrosamine',     '[N;!$(N=O)]-N=O'),
    ('Aromatic_nitro',  'c[N+](=O)[O-]'),
    ('Aliphatic_nitro', 'C[N+](=O)[O-]'),
    ('Ar_amine',        '[NH2]c'),
    ('Epoxide',         '[C;R0]1OC1'),
    ('Michael',         '[$(C=CC=O),$(C=CS)]'),
    ('Aziridine',       'C1CN1'),
    ('Diazonium',       '[#6][N+]#N'),
]

def has_any_alert(mol):
    if mol is None: return 0
    for name, smarts in ICH_M7_SMARTS:
        patt = Chem.MolFromSmarts(smarts)
        if patt and mol.HasSubstructMatch(patt): return 1
    return 0

df['sa_alert'] = [has_any_alert(mol) for mol in df['mol']]
sa_auc = roc_auc_score(y_ames, df['sa_alert'])
sa_sens = ((df['sa_alert']==1)&(y_ames==1)).sum() / max(1,y_ames.sum())
print(f'Structural Alerts baseline: AUC={sa_auc:.3f}  Sensitivity={sa_sens:.3f}')

# Random Forest QSAR (Method 2 of ICH M7 two-method framework)
rf_ames  = RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                                   class_weight='balanced', random_state=42)
ames_proba = cross_val_predict(rf_ames, X_full, y_ames, cv=cv5, method='predict_proba')[:,1]
ames_pred  = (ames_proba >= 0.5).astype(int)
ames_auc   = roc_auc_score(y_ames, ames_proba)
ames_mcc   = matthews_corrcoef(y_ames, ames_pred)
ames_sens  = ((ames_pred==1)&(y_ames==1)).sum() / max(1, y_ames.sum())
ames_spec  = ((ames_pred==0)&(y_ames==0)).sum() / max(1, (y_ames==0).sum())

# ICH M7 two-method integration
# Concordant positive (both methods positive) -> mutagenic
# Concordant negative (both negative) -> not mutagenic
# Discordant -> equivocal (expert review)
combined_pos = ((df['sa_alert']==1) | (ames_proba >= 0.5)).astype(int)
both_pos     = ((df['sa_alert']==1) & (ames_proba >= 0.5)).astype(int)

print(f'\nICH M7 Two-Method Assessment:')
print(f'  QSAR only:          AUC={ames_auc:.3f}  Sens={ames_sens:.3f}  Spec={ames_spec:.3f}')
print(f'  Either method +ve:  Sens={roc_auc_score(y_ames, combined_pos):.3f}')
print(f'  Both methods +ve:   Sens (conservative)={both_pos[y_ames==1].mean():.3f}')
print(f'  ICH M7 target:      Sensitivity >= 0.90')
ich_met = 'MEETS' if ames_sens >= 0.90 else 'BELOW'
print(f'  Status: {ich_met} ICH M7 sensitivity requirement')

In [ ]:
# ── 6.3 DILI (Drug-Induced Liver Injury) risk model ─────────────────────────
# DILI is the leading cause of drug withdrawal and regulatory rejection
# FDA DILIrank: most-concern / less-concern / no-concern

y_dili = df['DILI'].values
print(f'DILI concern: {y_dili.sum()}/{len(y_dili)} ({y_dili.mean()*100:.0f}%)')

# Known DILI risk structural alerts
DILI_SMARTS = [
    ('Quinone',           'O=C1C=CC(=O)C=C1'),
    ('Michael_acceptor',  '[$(C=CC=O)]'),
    ('Thio_carbonyl',     'C(=S)'),
    ('Nitroaro',          'c[N+](=O)[O-]'),
    ('Furan',             'c1ccoc1'),
    ('Thiophene',         'c1ccsc1'),
    ('BSEP_pharmacophore', 'C(=O)N1CCCC1'),  # cyclic amide
]

def dili_alert_score(mol):
    if mol is None: return 0
    return sum(1 for _, smarts in DILI_SMARTS
               if Chem.MolFromSmarts(smarts) and mol.HasSubstructMatch(Chem.MolFromSmarts(smarts)))

df['dili_alerts'] = [dili_alert_score(mol) for mol in df['mol']]

# Multi-model DILI comparison
dili_models = {
    'Random Forest':     RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=42),
    'Extra Trees':       ExtraTreesClassifier(n_estimators=300, class_weight='balanced', random_state=42),
}
print(f'\n{"Model":22s} {"AUC":>8} {"MCC":>8}')
print('-'*42)
dili_results = {}
for name, clf in dili_models.items():
    proba = cross_val_predict(clf, X_full, y_dili, cv=cv5, method='predict_proba')[:,1]
    auc = roc_auc_score(y_dili, proba)
    mcc = matthews_corrcoef(y_dili, (proba>=0.5).astype(int))
    dili_results[name] = {'auc':auc,'mcc':mcc,'proba':proba}
    print(f'{name:22s} {auc:8.3f} {mcc:8.3f}')

best_dili  = max(dili_results, key=lambda k: dili_results[k]['auc'])
best_proba = dili_results[best_dili]['proba']

# ── 6.4 LD50 acute toxicity model ─────────────────────────────────────────
y_ld50     = df['LD50_mgkg'].values
y_ld50_log = np.log10(y_ld50)

rf_ld50   = RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=42)
ld50_pred = cross_val_predict(rf_ld50, X_full, y_ld50_log, cv=KFold(5, shuffle=True, random_state=42))
ld50_r2   = r2_score(y_ld50_log, ld50_pred)
ld50_rmse = np.sqrt(mean_squared_error(y_ld50_log, ld50_pred))

print(f'\nLD50 Prediction: R2={ld50_r2:.3f}  RMSE={ld50_rmse:.3f} log units')

# GHS classification from predicted LD50
def ghs_class(ld50_mgkg):
    if ld50_mgkg <= 5:    return 'Cat 1 (Fatal)'
    elif ld50_mgkg <= 50: return 'Cat 2 (Fatal)'
    elif ld50_mgkg <= 300: return 'Cat 3 (Toxic)'
    elif ld50_mgkg <= 2000: return 'Cat 4 (Harmful)'
    else:                   return 'Cat 5 / Unclassified'

df['ghs_acute_oral'] = df['LD50_mgkg'].apply(ghs_class)
print('\nGHS Acute Oral Toxicity Distribution:')
print(df['ghs_acute_oral'].value_counts())

In [ ]:
# ── 6.5 Toxicity model visualisation ─────────────────────────────────────────
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# ROC curves for binary endpoints
ax = axes[0, 0]
binary_endpoints = [
    ('hERG', y_herg, herg_results[best_herg]['proba'], '#E74C3C'),
    ('Ames', y_ames, ames_proba, '#1565C0'),
    ('DILI', y_dili, best_proba, '#27AE60'),
    ('BBB',  y_bbb,  bbb_proba,  '#E67E22'),
]
for ep_name, y_true, y_score, col in binary_endpoints:
    fpr, tpr, _ = roc_curve(y_true, y_score)
    auc = roc_auc_score(y_true, y_score)
    ax.plot(fpr, tpr, color=col, lw=2.2, label=f'{ep_name} AUC={auc:.3f}')
ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC Curves -- Toxicity Classifiers', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# hERG probability distribution
ax = axes[0, 1]
herg_proba = herg_results[best_herg]['proba']
ax.hist(herg_proba[y_herg==0], bins=15, alpha=0.7, color='#1565C0', label='Non-active', density=True)
ax.hist(herg_proba[y_herg==1], bins=15, alpha=0.7, color='#E74C3C', label='hERG active', density=True)
ax.axvline(0.5, color='k', lw=2, linestyle='--', label='Threshold 0.5')
ax.set_xlabel('P(hERG active)'); ax.set_ylabel('Density')
ax.set_title('hERG Prediction Distribution', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# ICH M7 two-method concordance
ax = axes[0, 2]
ax.scatter(df['sa_alert']+np.random.uniform(-0.15,0.15,len(df)),
           ames_proba, c=['#E74C3C' if a==1 else '#1565C0' for a in y_ames],
           s=80, alpha=0.8, edgecolors='k', lw=0.5)
ax.axhline(0.5, color='k', lw=1.5, linestyle='--', alpha=0.6)
ax.set_xticks([0,1]); ax.set_xticklabels(['No SA alert','SA alert'])
ax.set_ylabel('P(Ames+) QSAR'); ax.set_title('ICH M7 Two-Method Concordance', fontweight='bold')
ax.text(0.05, 0.9, 'Both neg = Class 5', transform=ax.transAxes, fontsize=9, color='#1565C0')
ax.text(0.55, 0.9, 'Both pos = Class 2', transform=ax.transAxes, fontsize=9, color='#E74C3C')
ax.grid(True, alpha=0.3)

# LD50 observed vs predicted
ax = axes[1, 0]
ax.scatter(y_ld50_log, ld50_pred, c=['#E74C3C' if d==1 else '#1565C0' for d in y_dili],
           s=70, alpha=0.8, edgecolors='k', lw=0.5)
lims = [y_ld50_log.min()-0.2, y_ld50_log.max()+0.2]
ax.plot(lims, lims, 'k--', lw=1.5, alpha=0.6)
for ghs_val, ghs_name, col in [(np.log10(300),'Cat3/4 boundary','orange'),
                                 (np.log10(2000),'Cat4/5 boundary','green')]:
    ax.axvline(ghs_val, color=col, lw=1, linestyle=':', alpha=0.7, label=ghs_name)
ax.set_xlabel('Observed log10(LD50 mg/kg)'); ax.set_ylabel('Predicted')
ax.set_title(f'LD50 Prediction\nR2={ld50_r2:.3f}  RMSE={ld50_rmse:.3f}', fontweight='bold')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# DILI risk vs LogP
ax = axes[1, 1]
ax.scatter(logp_vals, df_phys['TPSA'].fillna(0),
           c=['#E74C3C' if d==1 else '#1565C0' for d in y_dili],
           s=80, alpha=0.85, edgecolors='k', lw=0.5)
ax.axvline(3, color='k', lw=1, linestyle='--', alpha=0.4)
ax.set_xlabel('LogP'); ax.set_ylabel('TPSA')
ax.set_title('DILI Risk in Chemical Space\n(red=DILI concern)', fontweight='bold')
ax.grid(True, alpha=0.3)

# GHS acute toxicity distribution
ax = axes[1, 2]
ghs_counts = df['ghs_acute_oral'].value_counts()
colours_ghs = ['#8B0000','#E74C3C','#E67E22','#F1C40F','#27AE60']
ax.barh(range(len(ghs_counts)), ghs_counts.values,
        color=colours_ghs[:len(ghs_counts)], alpha=0.85)
ax.set_yticks(range(len(ghs_counts)))
ax.set_yticklabels(ghs_counts.index, fontsize=9)
ax.set_xlabel('Count')
ax.set_title('GHS Acute Oral Toxicity\nClassification (50 drugs)', fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.suptitle('ADMET Toxicity Models', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Section 7 -- Multi-Endpoint ADMET Model & Stacked Ensemble

Rather than building separate models for each endpoint, a **stacked ensemble** trains
a meta-model on the outputs of multiple base learners, improving both accuracy and
uncertainty estimation.

```
Level 0 (base learners):    RF    GBM    ExtraTrees    XGBoost
                              |      |        |            |
Level 1 (meta-learner):     Logistic Regression / Ridge
                                        |
Output:                          Stacked prediction
```

In [ ]:
# ── 7.1 Multi-endpoint ADMET score calculator ──────────────────────────────
# Train final models on all data, then predict new compounds

# Train all models on full dataset (for deployment)
FINAL_MODELS = {
    'logS':   RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=42),
    'Papp':   RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42),
    'fup':    RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=42),
    'CLint':  GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42),
    'BBB':    RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42),
    'hERG':   RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42),
    'Ames':   RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42),
    'DILI':   GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=42),
    'LD50':   RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=42),
}

TARGETS = {
    'logS':  df['logS'].values,
    'Papp':  (df['Papp_Caco2'].values >= 10).astype(int),
    'fup':   np.log10(df['fup'].values.clip(0.001)),
    'CLint': np.log10(df['CLint'].values.clip(0.01)),
    'BBB':   df['BBB'].values,
    'hERG':  df['hERG'].values,
    'Ames':  df['Ames'].values,
    'DILI':  df['DILI'].values,
    'LD50':  np.log10(df['LD50_mgkg'].values),
}

print('Training all ADMET endpoint models...')
for ep, model in FINAL_MODELS.items():
    model.fit(X_full, TARGETS[ep])
    print(f'  {ep:8s}: trained on {len(TARGETS[ep])} compounds')

def predict_admet(smiles, name='Unknown'):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    mol = rdMolStandardize.LargestFragmentChooser().choose(mol)
    phys = physchem_descriptors(mol)
    fp   = morgan_fp(mol)
    X    = np.hstack([phys, np.array(maccs_fp(mol), dtype=np.float32)]).reshape(1,-1)
    # Rebuild phys+morgan feature
    X_pred = np.hstack([np.array(phys, dtype=np.float32).reshape(1,-1), fp.reshape(1,-1)])
    results = {'compound': name, 'smiles': smiles[:40]}
    # Regression endpoints
    for ep in ['logS','fup','CLint','LD50']:
        val = float(FINAL_MODELS[ep].predict(X_pred)[0])
        if ep == 'fup':   val = round(10**val, 4)
        elif ep == 'CLint': val = round(10**val, 2)
        elif ep == 'LD50':  val = round(10**val, 1)
        else: val = round(val, 3)
        results[ep] = val
    # Classification endpoints
    for ep in ['Papp','BBB','hERG','Ames','DILI']:
        prob = float(FINAL_MODELS[ep].predict_proba(X_pred)[0,1])
        results[f'P_{ep}'] = round(prob, 3)
        results[ep] = int(prob >= 0.5)
    # Derived metrics
    mw   = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    results['MW']    = round(mw, 1)
    results['LogP']  = round(logp, 2)
    results['TPSA']  = round(Descriptors.TPSA(mol), 1)
    results['QED']   = round(QED.qed(mol), 3)
    results['Ro5_OK'] = sum([mw>500, logp>5, rdMolDescriptors.CalcNumHBD(mol)>5,
                              rdMolDescriptors.CalcNumHBA(mol)>10]) <= 1
    # IVIVE estimate
    fup_v = results['fup']
    clint = results['CLint']
    Qh    = 90.0
    clint_L = clint * 45 * 1500 / 1000
    CLh     = (Qh * clint_L * fup_v) / (Qh + clint_L * fup_v)
    results['CLh_L_h']   = round(CLh, 2)
    results['t_half_h']  = round(0.693 * max(0.5,(0.2+0.8*logp)*70) / CLh, 1) if CLh > 0 else 99
    return results

# Test on 5 compounds
test_compounds = [
    ('Aspirin',           'CC(=O)Oc1ccccc1C(=O)O'),
    ('Atorvastatin',      'CC(C)c1c(C(=O)Nc2ccccc2F)c(-c2ccccc2)n(CC[C@@H](O)C[C@@H](O)CC(=O)O)c1-c1ccc(F)cc1'),
    ('Cisapride',         'COCCNC(=O)c1cc(Cl)c(N)cc1OC1CCNCC1'),
    ('NDMA',              'CN(C)N=O'),
    ('Caffeine',          'Cn1cnc2c1c(=O)n(C)c(=O)n2C'),
]

print('Multi-Endpoint ADMET Predictions:')
print('='*70)
for name, smi in test_compounds:
    r = predict_admet(smi, name)
    print(f'\n{name} ({r["MW"]} Da, LogP={r["LogP"]})')
    print(f'  Absorption: logS={r["logS"]}  P(Papp_high)={r["P_Papp"]}  Ro5={r["Ro5_OK"]}')
    print(f'  Distribution: fup={r["fup"]}  P(BBB)={r["P_BBB"]}  QED={r["QED"]}')
    print(f'  Metabolism: CLint={r["CLint"]} mL/min/mg  CLh={r["CLh_L_h"]} L/h  t1/2={r["t_half_h"]}h')
    print(f'  Toxicity: P(hERG)={r["P_hERG"]}  P(Ames)={r["P_Ames"]}  P(DILI)={r["P_DILI"]}')
    print(f'  LD50={r["LD50"]} mg/kg  GHS={ghs_class(r["LD50"])}')

---
## Section 8 -- Applicability Domain & Uncertainty Quantification

**Every ADMET prediction must include an applicability domain (AD) check.** A confident
wrong prediction is far worse than an uncertain correct one.

| AD method | Approach | When to use |
|-----------|---------|-------------|
| Tanimoto AD | Max similarity to training set | Standard, OECD GD 69 |
| Leverage / Williams plot | Hat matrix (H) | Linear models |
| Descriptor bounding box | Min/max of training features | Fast, interpretable |
| Conformal prediction | Guaranteed coverage sets | Regulatory submissions |
| Monte Carlo Dropout | Deep learning UQ | GNN models |

In [ ]:
# ── 8.1 Tanimoto-based applicability domain (OECD GD 69) ──────────────────
import numpy as np

class TanimotoAD:
    def __init__(self, percentile=5.0):
        self.percentile = percentile
        self.train_fps  = None
        self.threshold  = None

    def fit(self, fps):
        self.train_fps = fps.astype(np.float32)
        # Compute pairwise Tanimoto within training set
        ixj   = fps @ fps.T
        sx    = fps.sum(1, keepdims=True)
        union = sx + sx.T - ixj
        sim   = np.where(union > 0, ixj/union.astype(float), 0.0)
        np.fill_diagonal(sim, 0)
        max_sim = sim.max(1)
        self.threshold = float(np.percentile(max_sim, self.percentile))
        return self

    def max_similarity(self, test_fps):
        test  = test_fps.astype(np.float32)
        train = self.train_fps
        ixj   = test @ train.T
        st    = test.sum(1, keepdims=True)
        sv    = train.sum(1, keepdims=True).T
        union = st + sv - ixj
        return np.where(union > 0, ixj/union, 0.0).max(1)

    def in_ad(self, test_fps):
        return self.max_similarity(test_fps) >= self.threshold

ad = TanimotoAD(percentile=5)
ad.fit(morgan_fps)
print(f'AD threshold (5th percentile): {ad.threshold:.3f}')

# AD statistics
in_ad = ad.in_ad(morgan_fps)
max_tc = ad.max_similarity(morgan_fps)
print(f'Training compounds within AD: {in_ad.sum()}/{len(in_ad)} ({in_ad.mean()*100:.0f}%)')
print(f'Mean max-Tc: {max_tc.mean():.3f} +/- {max_tc.std():.3f}')

# ── 8.2 Conformal prediction for ADMET ──────────────────────────────────────
# Provides prediction sets with guaranteed coverage

class ConformalADMET:
    def __init__(self, model, alpha=0.10):
        self.model  = model
        self.alpha  = alpha
        self.cal_scores = None

    def calibrate(self, X_cal, y_cal):
        probs = self.model.predict_proba(X_cal)[:,1]
        self.cal_scores = np.array([1 - probs[i] if y_cal[i]==1 else probs[i]
                                    for i in range(len(y_cal))])
        return self

    def predict_set(self, X_test):
        probs = self.model.predict_proba(X_test)[:,1]
        n_cal = len(self.cal_scores)
        sets  = []
        for prob in probs:
            s = []
            for cls in [0, 1]:
                score = 1 - prob if cls==1 else prob
                pval  = (self.cal_scores >= score).sum() / (n_cal + 1)
                if pval > self.alpha: s.append(cls)
            sets.append(s)
        return sets

# Demo on hERG
from sklearn.model_selection import train_test_split
X_tr, X_cal, y_tr, y_cal = train_test_split(X_full, y_herg, test_size=0.3,
                                               stratify=y_herg, random_state=42)
rf_cp = RandomForestClassifier(n_estimators=200, random_state=42)
rf_cp.fit(X_tr, y_tr)
cp = ConformalADMET(rf_cp, alpha=0.10).calibrate(X_cal, y_cal)

X_te, y_te = X_full, y_herg
cp_sets    = cp.predict_set(X_te)
coverage   = np.mean([y_te[i] in cp_sets[i] for i in range(len(y_te))])
decisive   = sum(1 for s in cp_sets if len(s)==1) / len(cp_sets)

print(f'\nConformal Prediction (hERG, 90% level):')
print(f'  Coverage:       {coverage:.3f}  (guarantee >= 0.90)')
print(f'  Decisive (singleton): {decisive:.3f}')
print(f'  Uncertain (both classes): {sum(1 for s in cp_sets if len(s)==2)/len(cp_sets):.3f}')

In [ ]:
# ── 8.3 Feature importance and SHAP analysis ────────────────────────────────
# Train final RF hERG model on all data for feature importance
from sklearn.inspection import permutation_importance

rf_final = RandomForestClassifier(n_estimators=300, random_state=42)
rf_final.fit(X_phys, y_herg)  # use physchem only for interpretability

importances = rf_final.feature_importances_
top_idx     = np.argsort(importances)[::-1][:12]
top_names   = [PHYSCHEM_COLS[i] for i in top_idx]
top_vals    = importances[top_idx]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Feature importance
ax = axes[0]
ax.barh(range(len(top_names)), top_vals, color='#1565C0', alpha=0.85)
ax.set_yticks(range(len(top_names))); ax.set_yticklabels(top_names)
ax.set_xlabel('Feature Importance'); ax.set_title('hERG Model\nTop Features (RF)', fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# AD coverage by compound
ax = axes[1]
max_tc_all = ad.max_similarity(morgan_fps)
cols_ad = ['#27AE60' if v >= ad.threshold else '#E74C3C' for v in max_tc_all]
ax.bar(range(len(df)), max_tc_all, color=cols_ad, alpha=0.8)
ax.axhline(ad.threshold, color='k', lw=2, linestyle='--', label=f'AD threshold={ad.threshold:.3f}')
ax.set_xlabel('Compound index'); ax.set_ylabel('Max Tanimoto to training')
ax.set_title('Applicability Domain Check\n(green=within AD)', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Conformal prediction set sizes
ax = axes[2]
set_sizes = [len(s) for s in cp_sets]
colours_cp = ['#27AE60' if s==1 else '#E67E22' if s==2 else '#E74C3C' for s in set_sizes]
ax.bar(range(len(set_sizes)), set_sizes, color=colours_cp, alpha=0.8)
ax.set_xlabel('Compound index'); ax.set_ylabel('Prediction set size')
ax.set_title(f'Conformal Prediction Sets\n(decisive={decisive:.0%})', fontweight='bold')
ax.set_yticks([0,1,2])
ax.set_yticklabels(['Empty (anomaly)', 'Decisive', 'Uncertain'])
ax.grid(True, alpha=0.3)

plt.suptitle('Applicability Domain & Uncertainty Quantification', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Section 9 -- Drug Risk Assessment Pipeline & IVIVE

The complete pipeline connects ADMET predictions to regulatory risk assessment,
replacing animal experiments with a tiered computational approach aligned with
EPA HTTK, ICH M7(R2), and FDA Modernization Act 2.0.

```
SMILES Input
    |
    v
[1] Structural alerts (ICH M7, skin SA) ---- flags Class 1/2
    |
    v
[2] ADMET predictions (all 9 endpoints) ---- flags
    |
    v
[3] AD check (Tanimoto) ----------------- flags out-of-domain
    |
    v
[4] IVIVE (fup + CLint -> AED -> TTC) -- regulatory threshold
    |
    v
[5] IATA WoE risk tier + 3Rs report --- no-animal justification
```

In [ ]:
# ── 9.1 Complete drug risk assessment pipeline ─────────────────────────────

def drug_risk_assessment(smiles, compound_name='Unknown',
                          bioactivity_ec50_uM=1.0, dose_mg_day=100.0):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {'error': 'Invalid SMILES', 'compound': compound_name}

    # ── Step 1: Structural alerts ────────────────────────────────────────────
    sa_hits = [name for name, smarts in ICH_M7_SMARTS
               if Chem.MolFromSmarts(smarts) and mol.HasSubstructMatch(Chem.MolFromSmarts(smarts))]
    ich_m7_class = ('Class 1' if any('Nitrosamine' in h for h in sa_hits)
                    else 'Class 2' if sa_hits else 'Class 5')

    # ── Step 2: Full ADMET prediction ────────────────────────────────────────
    admet = predict_admet(smiles, compound_name)

    # ── Step 3: Applicability domain check ──────────────────────────────────
    std_mol = rdMolStandardize.LargestFragmentChooser().choose(mol)
    fp      = morgan_fp(std_mol).reshape(1, -1)
    max_tc  = float(ad.max_similarity(fp)[0])
    in_ad   = max_tc >= ad.threshold

    # ── Step 4: IVIVE (EPA HTTK) ────────────────────────────────────────────
    fup_v   = admet['fup']
    clint_v = admet['CLint']
    mw_v    = admet['MW']
    logp_v  = admet['LogP']
    # Well-stirred hepatic clearance
    Qh         = 90.0
    clint_L    = clint_v * 45 * 1500 / 1000
    CLh        = (Qh * clint_L * fup_v) / (Qh + clint_L * fup_v)
    # AED from EC50
    bw         = 70.0
    dose_umol  = (dose_mg_day * 1000 / mw_v) * 0.9 * 1e3  # nmol assuming 90% BA
    Vd         = max(0.5, (0.2 + 0.8 * logp_v) * bw)
    C0         = dose_umol / (Vd * 1000)  # uM
    Cmax_free  = C0 * fup_v
    AED_mg_kg  = (bioactivity_ec50_uM * CLh / 1000 * mw_v * 1440) / (bw * 1000)
    TTC_ug_day = AED_mg_kg * 1000 * bw * 1000  # ug/day

    # ── Step 5: Risk tier and 3Rs justification ─────────────────────────────
    concerns = []
    if sa_hits: concerns.append(f'ICH M7 {ich_m7_class}: {sa_hits}')
    if admet['P_hERG'] >= 0.5: concerns.append(f'hERG active (P={admet["P_hERG"]})')
    if admet['P_Ames'] >= 0.5: concerns.append(f'Ames mutagenicity (P={admet["P_Ames"]})')
    if admet['P_DILI'] >= 0.5: concerns.append(f'DILI risk (P={admet["P_DILI"]})')
    if not admet['Ro5_OK']:    concerns.append('Ro5 violation (poor oral BA expected)')
    if TTC_ug_day < 1.5:       concerns.append(f'TTC < 1.5 ug/day ({TTC_ug_day:.3f})')
    if not in_ad:              concerns.append(f'Outside AD (maxTc={max_tc:.3f})')

    n_concerns = len(concerns)
    risk_tier  = ('CRITICAL' if n_concerns >= 5 else
                  'HIGH'     if n_concerns >= 3 else
                  'MODERATE' if n_concerns >= 1 else 'LOW')

    return {
        'compound':          compound_name,
        'smiles':            smiles[:40],
        'risk_tier':         risk_tier,
        'n_concerns':        n_concerns,
        'concerns':          concerns,
        'ich_m7_class':      ich_m7_class,
        'sa_alerts':         sa_hits,
        'admet':             admet,
        'in_ad':             in_ad,
        'max_tc':            round(max_tc, 3),
        'ivive': {
            'CLh_L_h':       round(CLh, 2),
            'Cmax_free_uM':  round(Cmax_free, 4),
            'AED_mg_kg_day': round(AED_mg_kg, 5),
            'TTC_ug_day':    round(TTC_ug_day, 3),
            'TTC_flag':      TTC_ug_day < 1.5,
        },
        '3rs_justified':     risk_tier in ('LOW', 'MODERATE') and in_ad,
        'recommendation':    (
            'No animal studies required -- NAM data sufficient'
            if risk_tier in ('LOW', 'MODERATE') and in_ad
            else 'Follow-up in vitro or targeted animal study required'
        ),
    }

# Screen a library
SCREEN_LIBRARY = [
    ('Aspirin',       'CC(=O)Oc1ccccc1C(=O)O',          0.5),
    ('Caffeine',      'Cn1cnc2c1c(=O)n(C)c(=O)n2C',     5.0),
    ('Diclofenac',    'O=C(O)Cc1ccccc1Nc1c(Cl)cccc1Cl', 10.0),
    ('Cisapride',     'COCCNC(=O)c1cc(Cl)c(N)cc1OC1CCNCC1', 0.1),
    ('NDMA',          'CN(C)N=O',                         0.01),
    ('Metformin',     'CN(C)C(=N)NC(=N)N',              100.0),
    ('Atorvastatin',  'CC(C)c1c(C(=O)Nc2ccccc2F)c(-c2ccccc2)n(CC[C@@H](O)C[C@@H](O)CC(=O)O)c1-c1ccc(F)cc1', 0.3),
    ('Tamoxifen',     'CC/C(=C(\\c1ccccc1)/c1ccc(OCCN(C)C)cc1)c1ccccc1', 0.05),
]

print('Drug Risk Assessment Screening')
print('='*75)
risk_icons = {'LOW':'[GREEN]','MODERATE':'[YELLOW]','HIGH':'[ORANGE]','CRITICAL':'[RED]'}
results_all = []
for name, smi, ec50 in SCREEN_LIBRARY:
    r = drug_risk_assessment(smi, name, ec50, dose_mg_day=100)
    results_all.append(r)
    icon = risk_icons[r['risk_tier']]
    print(f'\n{icon} {name:18s}  Risk: {r["risk_tier"]:8s}  ({r["n_concerns"]} concerns)')
    print(f'   ADMET: logS={r["admet"]["logS"]}  fup={r["admet"]["fup"]}  '
          f'P(hERG)={r["admet"]["P_hERG"]}  P(Ames)={r["admet"]["P_Ames"]}  P(DILI)={r["admet"]["P_DILI"]}')
    print(f'   IVIVE: Cmax_free={r["ivive"]["Cmax_free_uM"]} uM  TTC={r["ivive"]["TTC_ug_day"]} ug/day')
    if r['concerns']:
        print(f'   Concerns: {r["concerns"][:2]}')
    print(f'   3Rs: {r["recommendation"][:55]}')

---
## Section 10 -- ADMET Dashboard & 3Rs Report Generation

In [ ]:
# ── 10.1 Complete ADMET dashboard ──────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import numpy as np

def admet_dashboard(results_list, title='ADMET Drug Risk Screening'):
    n = len(results_list)
    fig = plt.figure(figsize=(22, 16))
    gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.50, wspace=0.40)

    RISK_COLS = {'LOW':'#27AE60','MODERATE':'#F39C12','HIGH':'#E74C3C','CRITICAL':'#8B0000'}

    names   = [r['compound'] for r in results_list]
    risks   = [r['risk_tier'] for r in results_list]
    n_conc  = [r['n_concerns'] for r in results_list]

    # Panel 1: Risk tier summary
    ax1 = fig.add_subplot(gs[0, 0])
    bar_cols = [RISK_COLS[r] for r in risks]
    ax1.barh(names, n_conc, color=bar_cols, alpha=0.85)
    ax1.set_xlabel('Number of Concerns')
    ax1.set_title('Risk Tier by Compound', fontweight='bold')
    for name, col, risk in zip(names, bar_cols, risks):
        pass
    patches = [mpatches.Patch(color=v, label=k) for k,v in RISK_COLS.items()]
    ax1.legend(handles=patches, fontsize=8, loc='lower right')
    ax1.grid(True, alpha=0.3, axis='x')

    # Panel 2: Toxicity probability heatmap
    ax2 = fig.add_subplot(gs[0, 1:3])
    tox_endpoints = ['P_hERG','P_Ames','P_DILI']
    tox_labels    = ['hERG','Ames','DILI']
    tox_matrix    = np.array([[r['admet'][ep] for ep in tox_endpoints] for r in results_list])
    im2 = ax2.imshow(tox_matrix.T, cmap='RdYlGn_r', vmin=0, vmax=1, aspect='auto')
    ax2.set_xticks(range(n)); ax2.set_xticklabels(names, rotation=30, ha='right', fontsize=9)
    ax2.set_yticks(range(3)); ax2.set_yticklabels(tox_labels)
    for i in range(3):
        for j in range(n):
            val = tox_matrix[j, i]
            ax2.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8,
                     color='white' if val > 0.6 else 'black')
    plt.colorbar(im2, ax=ax2, label='Probability')
    ax2.set_title('Toxicity Endpoint Probabilities', fontweight='bold')

    # Panel 3: ADMET radar chart for best/worst
    ax3 = fig.add_subplot(gs[0, 3])
    admet_metrics = ['logS','LogP','QED','fup','CLint']
    best_compound  = min(results_list, key=lambda r: r['n_concerns'])
    worst_compound = max(results_list, key=lambda r: r['n_concerns'])
    best_vals  = [best_compound['admet'].get(m, 0) for m in admet_metrics]
    worst_vals = [worst_compound['admet'].get(m, 0) for m in admet_metrics]
    x = np.arange(len(admet_metrics))
    ax3.plot(x, best_vals,  'o-', color='#27AE60', lw=2, ms=7, label=best_compound['compound'])
    ax3.plot(x, worst_vals, 'o-', color='#E74C3C', lw=2, ms=7, label=worst_compound['compound'])
    ax3.set_xticks(x); ax3.set_xticklabels(admet_metrics, fontsize=9)
    ax3.set_title('Best vs Worst ADMET', fontweight='bold')
    ax3.legend(fontsize=8); ax3.grid(True, alpha=0.3)

    # Panel 4: Solubility vs Permeability (BCS classification)
    ax4 = fig.add_subplot(gs[1, 0:2])
    logS_pred  = [r['admet']['logS'] for r in results_list]
    papp_pred  = [r['admet']['P_Papp'] for r in results_list]
    risk_cols  = [RISK_COLS[r['risk_tier']] for r in results_list]
    ax4.scatter(logS_pred, papp_pred, c=risk_cols, s=100, alpha=0.85, edgecolors='k', lw=0.8)
    ax4.axvline(-4, color='k', lw=1.2, linestyle='--', alpha=0.5, label='LogS=-4 (BCS boundary)')
    ax4.axhline(0.5, color='k', lw=1.2, linestyle='--', alpha=0.5, label='P(Papp)=0.5')
    for r in results_list:
        ax4.annotate(r['compound'][:6], (r['admet']['logS'], r['admet']['P_Papp']), fontsize=7)
    ax4.set_xlabel('Predicted logS'); ax4.set_ylabel('P(High Papp / Caco-2)')
    ax4.set_title('BCS Chemical Space\n(I=top-right, IV=bottom-left)', fontweight='bold')
    ax4.legend(fontsize=8); ax4.grid(True, alpha=0.3)

    # Panel 5: IVIVE TTC comparison
    ax5 = fig.add_subplot(gs[1, 2:])
    ttc_vals = [r['ivive']['TTC_ug_day'] for r in results_list]
    bar_cols5 = ['#E74C3C' if v < 1.5 else '#27AE60' for v in ttc_vals]
    bars = ax5.bar(names, ttc_vals, color=bar_cols5, alpha=0.85)
    ax5.axhline(1.5, color='k', lw=2.5, linestyle='--', label='ICH M7 TTC 1.5 ug/day')
    ax5.set_yscale('log')
    ax5.set_ylabel('Estimated AED (ug/day)')
    ax5.tick_params(axis='x', rotation=30)
    ax5.set_title('IVIVE: TTC Comparison\n(red=below ICH M7 threshold)', fontweight='bold')
    ax5.legend(fontsize=9); ax5.grid(True, alpha=0.3)

    # Panel 6: Summary table
    ax6 = fig.add_subplot(gs[2, :])
    ax6.axis('off')
    table_data = []
    for r in results_list:
        a = r['admet']
        row = [
            r['compound'],
            r['risk_tier'],
            str(a['logS']),
            f"{a['P_Papp']:.2f}",
            str(a['fup']),
            str(a['CLint']),
            f"{a['P_hERG']:.2f}",
            f"{a['P_Ames']:.2f}",
            f"{a['P_DILI']:.2f}",
            str(a.get('LD50', 'N/A')),
            'YES' if r['3rs_justified'] else 'NO',
        ]
        table_data.append(row)
    col_labels = ['Compound','Risk','logS','P(Papp)','fup','CLint','P(hERG)','P(Ames)','P(DILI)','LD50','3Rs OK']
    tbl = ax6.table(cellText=table_data, colLabels=col_labels,
                    cellLoc='center', loc='center', bbox=[0,0,1,1])
    tbl.auto_set_font_size(False); tbl.set_fontsize(8.5)
    for j in range(len(col_labels)):
        tbl[0,j].set_facecolor('#1565C0'); tbl[0,j].set_text_props(color='white',fontweight='bold')
    for i in range(1, len(table_data)+1):
        risk = table_data[i-1][1]
        bg = {'LOW':'#E8F8F0','MODERATE':'#FEF9E7','HIGH':'#FDEDEC','CRITICAL':'#F1948A'}.get(risk,'white')
        for j in range(len(col_labels)): tbl[i,j].set_facecolor(bg)
    ax6.set_title('Complete ADMET Risk Assessment Summary', fontweight='bold', pad=10)

    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.01)
    plt.savefig('admet_risk_dashboard.png', dpi=130, bbox_inches='tight')
    plt.show()
    print('Dashboard saved: admet_risk_dashboard.png')

admet_dashboard(results_all, 'ADMET Drug Risk Assessment -- 8 Compound Screening Library')

In [ ]:
# ── 10.2 3Rs regulatory report generator ────────────────────────────────────

def generate_3rs_admet_report(results):
    r = results
    a = r['admet']
    iv = r['ivive']
    lines = [
        f'3Rs ADMET RISK ASSESSMENT REPORT',
        f'Compound:    {r["compound"]}',
        f'SMILES:      {r["smiles"]}',
        f'Risk Tier:   {r["risk_tier"]}',
        f'AD Status:   {"Within AD (maxTc=" + str(r["max_tc"]) + ")" if r["in_ad"] else "OUTSIDE AD (predictions unreliable)"}',
        '',
        'PHYSICOCHEMICAL PROFILE',
        f'  MW={a["MW"]} Da | LogP={a["LogP"]} | TPSA={a["TPSA"]} A2 | QED={a["QED"]} | Ro5={a["Ro5_OK"]}',
        '',
        'ABSORPTION',
        f'  logS={a["logS"]} (solubility class: {solubility_class(a["logS"])})',
        f'  P(high Papp Caco-2): {a["P_Papp"]}  ->  {"HIGH permeability" if a["P_Papp"]>=0.5 else "LOW permeability"}',
        f'  Veber rules: {"PASS" if a["Ro5_OK"] else "FAIL"}',
        '',
        'DISTRIBUTION',
        f'  fup={a["fup"]} (plasma protein binding: {"low" if a["fup"]<0.1 else "moderate" if a["fup"]<0.4 else "high"} unbound)',
        f'  P(BBB): {a["P_BBB"]}  ->  {"CNS penetrant" if a["P_BBB"]>=0.5 else "Non-CNS"}',
        '',
        'METABOLISM',
        f'  CLint={a["CLint"]} mL/min/mg  (stability: {clint_class(a["CLint"])})',
        f'  CLh (hepatic)={iv["CLh_L_h"]} L/h  |  t1/2={a["t_half_h"]}h',
        '',
        'TOXICITY',
        f'  P(hERG): {a["P_hERG"]}  |  P(Ames): {a["P_Ames"]}  |  P(DILI): {a["P_DILI"]}',
        f'  LD50: ~{a["LD50"]} mg/kg  ->  GHS: {ghs_class(a["LD50"])}',
        f'  ICH M7: {r["ich_m7_class"]}  (alerts: {r["sa_alerts"] or "none"})',
        '',
        'IVIVE (EPA HTTK)',
        f'  Cmax_free: {iv["Cmax_free_uM"]} uM  |  AED: {iv["AED_mg_kg_day"]} mg/kg/day',
        f'  TTC: {iv["TTC_ug_day"]} ug/day  {"WARNING: below 1.5 ug/day threshold!" if iv["TTC_flag"] else "(OK)"}',
        '',
        'CONCERNS',
    ]
    for concern in r['concerns']:
        lines.append(f'  * {concern}')
    if not r['concerns']:
        lines.append('  No concerns identified')
    lines += [
        '',
        'REGULATORY FRAMEWORKS APPLIED',
        '  ICH M7(R2) 2023   -- two-method genotoxicity (SA + QSAR)',
        '  ICH E14/S7B 2022  -- hERG/CiPA cardiac safety',
        '  EPA HTTK IVIVE    -- in vitro to in vivo extrapolation',
        '  OECD TG 497       -- skin sensitisation (if skin SA present)',
        '  FDA Mod. Act 2.0  -- NAM data accepted for regulatory submissions',
        '',
        f'3Rs CONCLUSION: {r["recommendation"]}',
        f'Animal studies justified: {not r["3rs_justified"]}',
    ]
    return '\n'.join(lines)

# Generate report for cisapride (high-risk compound)
cisapride_result = next(r for r in results_all if 'Cisapride' in r['compound'])
print(generate_3rs_admet_report(cisapride_result))
print('\n' + '='*65)
aspirin_result = next(r for r in results_all if 'Aspirin' in r['compound'])
print(generate_3rs_admet_report(aspirin_result))

In [ ]:
# ── 10.3 GNN template (graph neural networks for ADMET) ─────────────────────
print('GNN ADMET Model Template (requires PyTorch Geometric)')
print('Install: pip install torch-geometric torch-scatter torch-sparse')
print()

GNN_TEMPLATE = '''
# ── Graph Neural Network for ADMET prediction ────────────────────────────────
# Reference: Wieder 2020 (J Cheminformatics), Yang 2019 (DMPNN)
# Better than fingerprints for: pKa, metabolic stability, solubility

import torch
import torch.nn as nn
from torch_geometric.nn import GCNConv, GATConv, GlobalAttention
from torch_geometric.data import Data, DataLoader

class ADMETGraphNet(nn.Module):
    def __init__(self, node_dim=9, edge_dim=3, hidden=128, n_tasks=9):
        super().__init__()
        # Encoder: 3 GCN layers with batch norm
        self.conv1 = GCNConv(node_dim, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.conv3 = GCNConv(hidden, hidden)
        self.bn1   = nn.BatchNorm1d(hidden)
        self.bn2   = nn.BatchNorm1d(hidden)
        # Global readout: attention pooling
        self.pool  = GlobalAttention(gate_nn=nn.Linear(hidden, 1))
        # Multi-task head (one output per ADMET endpoint)
        self.head  = nn.Sequential(
            nn.Linear(hidden, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, n_tasks)
        )
    def forward(self, data):
        x = data.x.float()
        x = torch.relu(self.bn1(self.conv1(x, data.edge_index)))
        x = torch.relu(self.bn2(self.conv2(x, data.edge_index)))
        x = torch.relu(self.conv3(x, data.edge_index))
        x = self.pool(x, data.batch)
        return self.head(x)  # [batch_size, n_tasks]

# Node features (9 per atom): atomic_num, degree, formal_charge, hybridisation,
#   aromatic, H_count, ring, chirality, electronegativity
# Edge features (3 per bond): bond_type, ring_bond, stereo

# Multi-task loss: BCE for classification, MSE for regression
def multi_task_loss(pred, targets, task_types):
    total = 0
    for i, (preds, tgt, ttype) in enumerate(zip(pred.T, targets.T, task_types)):
        mask = ~torch.isnan(tgt)
        if mask.sum() == 0: continue
        if ttype == "cls":
            total += nn.BCEWithLogitsLoss()(preds[mask], tgt[mask])
        else:
            total += nn.MSELoss()(preds[mask], tgt[mask])
    return total / len(task_types)
'''
print(GNN_TEMPLATE)

In [ ]:
# ── 10.4 Complete cheatsheet ────────────────────────────────────────────────
lines = [
    'ADMET MODEL BUILDING -- INDUSTRY STANDARD REFERENCE',
    '',
    'FEATURES',
    '  Best combo:     Physicochemical (26) + Morgan ECFP4 (2048)',
    '  Physicochemical: MW, LogP, TPSA, HBD, HBA, Fsp3, QED, nN, nO, basicN',
    '  Fingerprints:   Morgan r=2, 2048 bits (ECFP4)',
    '  Supplementary:  MACCS keys (166), RDKit path FP (2048)',
    '',
    'MODELS BY ENDPOINT',
    '  LogS solubility:  RF / GBM regression (RMSE < 1.0)',
    '  Caco-2 Papp:      RF classifier (AUC > 0.80)',
    '  fup:              RF regression on log10(fup) (R2 > 0.60)',
    '  BBB:              RF classifier + CNS MPO (AUC > 0.80)',
    '  CLint:            GBM regression on log10(CLint) (R2 > 0.60)',
    '  hERG:             RF/GBM classifier (AUC > 0.85, ICH E14)',
    '  Ames:             RF classifier (Sens >= 0.90 per ICH M7)',
    '  DILI:             GBM classifier (AUC > 0.80)',
    '  LD50:             RF regression on log10(LD50) (R2 > 0.60)',
    '',
    'VALIDATION (OECD GD 69)',
    '  Cross-validation: StratifiedKFold(5) for classifiers, KFold(5) for regression',
    '  External test set: scaffold split or temporal split recommended',
    '  Metrics: AUC + MCC (class), R2 + RMSE (regression)',
    '  ICH M7: sensitivity >= 90% mandatory for Ames model',
    '',
    'APPLICABILITY DOMAIN',
    '  Method: Tanimoto AD (max-Tc to training), threshold at 5th percentile',
    '  UQ: Conformal prediction sets (guaranteed 90% coverage)',
    '  Flag: any compound with maxTc < threshold -> outside AD',
    '',
    'IVIVE PIPELINE (EPA HTTK)',
    '  fup (RED assay) + CLint (HLM) -> CLh (well-stirred) -> Css -> AED',
    '  TTC threshold: 1.5 ug/day (ICH M7 Class 2/3)',
    '  Formula: CLh = Qh*CLint*fup / (Qh + CLint*fup)',
    '',
    'REGULATORY FRAMEWORKS COVERED',
    '  ICH M7(R2) 2023:    Ames SA + QSAR (two-method), sensitivity >= 90%',
    '  ICH E14/S7B 2022:   hERG + CiPA (replace in vivo QT)',
    '  EPA HTTK:           IVIVE fup+CLint -> AED (replace dose-range animal)',
    '  OECD TG 497 DA2:    Skin SA + in vitro (replace LLNA mouse)',
    '  OECD GD 69:         5 QSAR principles for model validity',
    '  FDA Mod. Act 2.0:   NAM/ADMET data accepted in IND submissions',
    '',
    'PRODUCTION CHECKLIST',
    '  Standardise SMILES: LargestFragmentChooser + Uncharger',
    '  Check AD before EVERY prediction (Tanimoto + descriptor bounding box)',
    '  Use log10 transform for right-skewed endpoints (LD50, CLint, fup)',
    '  Balance classes: class_weight=balanced for hERG, Ames, DILI',
    '  Conformal prediction for regulatory submissions (coverage guarantee)',
    '  Always report OECD QMRF (QSAR Model Reporting Format)',
]
print('\n'.join(lines))